# TrueID Live Commerce Copilot - Recording Demo

This notebook keeps the main demo focused on recording-mode ASR and commerce actions. Catalog management and standalone realtime/live-mic fallbacks are provided in separate cells below.


In [ ]:
import contextlib
import io
import os
import sys
import subprocess
from pathlib import Path

from IPython.display import HTML, display

REPO_URL = "https://github.com/Siratish/Live-Commerce-Copilot.git"
REPO_DIR_NAME = "Live-Commerce-Copilot"


def setup_card(title, detail=""):
    display(HTML(
        '<div style="border:1px solid #bfdbfe;background:#eff6ff;color:#1e3a8a;'
        'border-radius:8px;padding:12px;font-family:Arial,sans-serif;margin-bottom:10px;">'
        f'<strong>{title}</strong>'
        f'<div style="font-size:13px;line-height:1.45;margin-top:4px;">{detail}</div>'
        '</div>'
    ))


def looks_like_project(path: Path) -> bool:
    return (path / "config" / "demo.yaml").exists() and (path / "src").exists()


repo_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if looks_like_project(candidate):
        repo_root = candidate
        break

if repo_root is None:
    clone_parent = Path("/content") if Path("/content").exists() else Path.cwd()
    repo_root = clone_parent / REPO_DIR_NAME
    if repo_root.exists() and not looks_like_project(repo_root):
        raise RuntimeError(f"{repo_root} exists but does not look like the target project.")
    if not repo_root.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(repo_root)], stdout=subprocess.DEVNULL)

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

requirements = repo_root / "requirements.txt"
if requirements.exists():
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)]
    )

try:
    import ipywidgets  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ipywidgets"])

from src.utils.live_mic import preload_openai_whisper_model

with contextlib.redirect_stdout(io.StringIO()):
    PRELOADED_OPENAI_WHISPER_TURBO = preload_openai_whisper_model("turbo")

setup_card(
    "Notebook setup complete",
    f"Repo ready at <code>{repo_root}</code>. OpenAI Whisper turbo is loaded for the demo.",
)


In [ ]:
from pathlib import Path
from src.utils.full_demo_ui import display_full_demo_ui

display_full_demo_ui(Path.cwd())


## Catalog manager

Run this cell when you want to review products or promotions separately from the recording demo. Products and promotions are shown one list at a time, and product additions can include an uploaded image.


In [ ]:
# CATALOG_MANAGER_CELL_V1
from pathlib import Path
from src.utils.full_demo_ui import display_catalog_manager_ui

display_catalog_manager_ui(Path.cwd())


## Standalone fallback: realtime audio-file stream

If the full control-room UI gets in the way, run this cell instead. It gives you a small source picker and then calls `run_realtime_audio_file_demo` directly, using the same realtime stream UI that has been working independently.


In [ ]:
# STANDALONE_FALLBACK_CELLS_V1
# Simple fallback UI: choose Audio 1, Audio 2, or upload a file, then run the known-good
# realtime audio-file stream demo directly.
from pathlib import Path
import time

import ipywidgets as widgets
from IPython.display import HTML, display, clear_output

from src.utils.realtime_audio_file import (
    RealtimeAudioFileDemoConfig,
    run_realtime_audio_file_demo,
)

FALLBACK_REPO_ROOT = Path.cwd()
FALLBACK_SAMPLE_AUDIO = {
    "Audio 1 - Beauty": FALLBACK_REPO_ROOT / "data" / "demo" / "audio" / "1.mp3",
    "Audio 2 - Tech": FALLBACK_REPO_ROOT / "data" / "demo" / "audio" / "2.mp3",
}
FALLBACK_UPLOAD_DIR = FALLBACK_REPO_ROOT / "outputs" / "standalone_fallback" / "uploads"
FALLBACK_UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

fallback_status = widgets.HTML()
fallback_runner = widgets.Output()


def _fallback_card(title, detail="", kind="info"):
    colors = {
        "info": ("#eff6ff", "#bfdbfe", "#1e3a8a"),
        "done": ("#f0fdf4", "#bbf7d0", "#14532d"),
        "error": ("#fef2f2", "#fecaca", "#7f1d1d"),
    }
    bg, border, text = colors.get(kind, colors["info"])
    return (
        f'<div style="border:1px solid {border};background:{bg};color:{text};'
        'border-radius:8px;padding:12px;font-family:Arial,sans-serif;">'
        f'<strong>{title}</strong>'
        f'<div style="font-size:13px;line-height:1.45;margin-top:4px;">{detail}</div>'
        '</div>'
    )


def _first_upload_item(upload_value):
    if not upload_value:
        return None, None
    if isinstance(upload_value, dict):
        name, item = next(iter(upload_value.items()))
        return name, item
    item = upload_value[0]
    return item.get("name", "uploaded_audio"), item


def _save_uploaded_audio(upload_widget: widgets.FileUpload) -> Path:
    name, item = _first_upload_item(upload_widget.value)
    if item is None:
        raise RuntimeError("No uploaded audio file found.")
    content = item.get("content")
    if content is None:
        raise RuntimeError("Uploaded audio file has no content payload.")
    safe_name = "".join(ch if ch.isalnum() or ch in ".-_" else "_" for ch in str(name))
    if not safe_name:
        safe_name = f"uploaded_audio_{int(time.time())}.wav"
    path = FALLBACK_UPLOAD_DIR / safe_name
    path.write_bytes(bytes(content))
    return path


def run_standalone_realtime_file(audio_path: Path, label: str):
    if not audio_path.exists():
        fallback_status.value = _fallback_card("Audio file not found", str(audio_path), "error")
        return

    safe_label = "".join(ch.lower() if ch.isalnum() else "_" for ch in label).strip("_") or "source"
    output_dir = FALLBACK_REPO_ROOT / "outputs" / "standalone_fallback" / "realtime_audio_file" / safe_label
    config = RealtimeAudioFileDemoConfig(
        audio_path=audio_path,
        chunk_seconds=15.0,
        dynamic_chunking=True,
        min_chunk_seconds=1.0,
        pause_seconds=0.3,
        silence_threshold=0.015,
        max_chunks=None,
        language="th",
        asr_provider="openai_whisper",
        asr_model="turbo",
        install_asr_deps=False,
        output_dir=output_dir,
        catalog_path=FALLBACK_REPO_ROOT / "data" / "demo" / "product_catalog.csv",
        promotions_path=FALLBACK_REPO_ROOT / "data" / "demo" / "promotions.csv",
    )
    source_picker.layout.display = "none"
    fallback_status.value = _fallback_card(
        "Realtime stream ready",
        "Use the play/pause button inside the generated stream panel. Source choices return after the run finishes.",
        "info",
    )
    with fallback_runner:
        clear_output(wait=True)
        try:
            summary = run_realtime_audio_file_demo(config)
            fallback_status.value = _fallback_card(
                "Realtime audio-file demo finished",
                f"{summary.get('caption_count', 0)} captions ? {summary.get('action_count', 0)} actions<br>"
                f"Captions: <code>{summary.get('captions_json', '')}</code><br>"
                f"Actions: <code>{summary.get('actions_json', '')}</code>",
                "done",
            )
        except Exception as exc:
            fallback_status.value = _fallback_card("Realtime audio-file demo failed", str(exc), "error")
        finally:
            source_picker.layout.display = ""


audio1_button = widgets.Button(
    description="Audio 1",
    icon="play",
    button_style="danger",
    layout=widgets.Layout(width="132px", height="44px"),
)
audio2_button = widgets.Button(
    description="Audio 2",
    icon="play",
    button_style="danger",
    layout=widgets.Layout(width="132px", height="44px"),
)
upload_widget = widgets.FileUpload(
    accept="audio/*",
    multiple=False,
    description="Upload audio",
    layout=widgets.Layout(width="190px", height="44px"),
)

audio1_button.on_click(
    lambda _button: run_standalone_realtime_file(
        FALLBACK_SAMPLE_AUDIO["Audio 1 - Beauty"],
        "Audio 1 - Beauty",
    )
)
audio2_button.on_click(
    lambda _button: run_standalone_realtime_file(
        FALLBACK_SAMPLE_AUDIO["Audio 2 - Tech"],
        "Audio 2 - Tech",
    )
)


def _on_upload_change(change):
    if not change.get("new"):
        return
    try:
        audio_path = _save_uploaded_audio(upload_widget)
    except Exception as exc:
        fallback_status.value = _fallback_card("Upload failed", str(exc), "error")
        return
    run_standalone_realtime_file(audio_path, f"Uploaded - {audio_path.name}")


upload_widget.observe(_on_upload_change, names="value")
source_picker = widgets.HBox([audio1_button, audio2_button, upload_widget], layout=widgets.Layout(gap="10px", flex_flow="row wrap"))

display(
    widgets.VBox(
        [
            widgets.HTML("<h3>Standalone realtime audio-file demo</h3>"),
            source_picker,
            fallback_status,
            fallback_runner,
        ],
        layout=widgets.Layout(gap="10px"),
    )
)


## Standalone fallback: live microphone stream

Run this cell when you want the direct `run_colab_live_mic_demo` panel without the full demo wrapper.


In [ ]:
# STANDALONE_FALLBACK_CELLS_V1
# Direct fallback cell: run the known-good live mic demo without the recording/catalog UIs.
from pathlib import Path

from IPython.display import HTML, display

from src.utils.live_mic import LiveMicDemoConfig, run_colab_live_mic_demo

LIVE_MIC_FALLBACK_CONFIG = LiveMicDemoConfig(
    chunk_seconds=15.0,
    min_chunk_seconds=1.0,
    pause_seconds=0.3,
    silence_threshold=0.015,
    max_chunks=None,
    language="th",
    asr_provider="openai_whisper",
    asr_model="turbo",
    install_asr_deps=False,
    show_debug_panel=True,
    continuous_recording=True,
    max_queue_chunks=16,
    output_dir=Path("outputs/standalone_fallback/live_mic"),
    catalog_path=Path("data/demo/product_catalog.csv"),
    promotions_path=Path("data/demo/promotions.csv"),
)

display(HTML(
    '<div style="border:1px solid #bfdbfe;background:#eff6ff;color:#1e3a8a;'
    'border-radius:8px;padding:12px;font-family:Arial,sans-serif;margin-bottom:10px;">'
    '<strong>Standalone live mic demo</strong>'
    '<div style="font-size:13px;line-height:1.45;margin-top:4px;">'
    'Use the mic button in the panel below to start/pause/resume recording. '
    'Interrupt this cell when you are done.'
    '</div></div>'
))

live_mic_fallback_summary = run_colab_live_mic_demo(LIVE_MIC_FALLBACK_CONFIG)
display(HTML(
    '<div style="border:1px solid #bbf7d0;background:#f0fdf4;color:#14532d;'
    'border-radius:8px;padding:12px;font-family:Arial,sans-serif;margin-top:10px;">'
    '<strong>Live mic demo finished</strong>'
    f'<div style="font-size:13px;line-height:1.45;margin-top:4px;">'
    f'{live_mic_fallback_summary.get("caption_count", 0)} captions ? '
    f'{live_mic_fallback_summary.get("action_count", 0)} actions<br>'
    f'Captions: <code>{live_mic_fallback_summary.get("captions_json", "")}</code><br>'
    f'Actions: <code>{live_mic_fallback_summary.get("actions_json", "")}</code>'
    '</div></div>'
))
